# 🍔 Food Delivery Demand Forecasting

This notebook contains the complete end-to-end pipeline for predicting food delivery times (`Time_taken(min)`) using temporal features. The goal is to optimize delivery operations and improve internal logistics by identifying peak periods and accurately estimated arrival times.

### 1. Environment Setup & Data Loading

We initialize the environment by ensuring all dependencies are installed and then load the raw order data from the `Data/` directory.

In [ ]:
%pip install scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the dataset
df = pd.read_csv('Data/orders.csv')
print(f"Initial dataset shape: {df.shape}")
df.head()

### 2. Data Cleaning & Feature Engineering

Temporal features are critical for demand forecasting. We convert raw time strings into usable datetime objects and extract numerical representations of hours, days, and months.

In [ ]:
# Drop records with missing vital information
df = df.dropna()

# Parse temporal columns
df['Time_Orderd'] = pd.to_datetime(df['Time_Orderd'], errors='coerce')
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True, errors='coerce')

# Engineer temporal features
df['hour'] = df['Time_Orderd'].dt.hour
df['day'] = df['Order_Date'].dt.dayofweek
df['month'] = df['Order_Date'].dt.month
df['weekend'] = df['day'].apply(lambda x: 1 if x >= 5 else 0)

# Clean the target variable (Time_taken(min)) by extracting numerical values from strings
df['Time_taken(min)'] = df['Time_taken(min)'].str.extract('(\d+)').astype(float)

# Final cleanup of newly generated NaNs from parsing errors
df = df.dropna()
print(f"Processed dataset shape: {df.shape}")
df.info()

### 3. Exploratory Data Analysis (EDA)

Understanding the distribution of orders over time helps validate our feature engineering and identify business patterns (e.g., lunch/dinner rushes).

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(x='hour', data=df, palette='viridis')
plt.title("Order Volume by Hour of Day")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(10, 6))
sns.countplot(x='day', data=df, palette='magma')
plt.title("Order Volume by Day of Week (0=Monday, 6=Sunday)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 4. Predictive Modeling: Random Forest Regressor

We use a Random Forest model due to its robustness against non-linear relationships and its ability to handle categorical temporal features effectively.

In [ ]:
# Feature selection
X = df[['hour', 'day', 'month', 'weekend']]
y = df['Time_taken(min)']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the regressor
model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Generate predictions
preds = model.predict(X_test)

# Performance Evaluation
print("--- Model Performance Summary ---")
print(f"Mean Squared Error (MSE): {mean_squared_error(y_test, preds):.2f}")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, preds):.2f} minutes")
print(f"R² Score: {r2_score(y_test, preds):.4f}")

### 5. Feature Importance Analysis

Finally, we analyze which temporal factors have the most weight in determining delivery durations. This provides actionable insights for logistics planning.

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
importance.sort_values(ascending=False).plot(kind='barh', color='teal')
plt.title("Impact of Temporal Features on Delivery Time")
plt.xlabel("Relative Importance Score")
plt.show()

# Export the final preprocessed dataset for dashboarding (e.g., Tableau)
df.to_csv('Data/cleaned_data.csv', index=False)
print("Pipeline complete. Cleaned data saved to 'Data/cleaned_data.csv'.")